# 05 — Problemas de Satisfacción de Restricciones

**Curso:** Inteligencia Artificial — Optimización

Este notebook contiene ejemplos guiados y actividades para desarrollar en clase.

## Objetivos

- representar un CSP mediante variables, dominios y restricciones;
- resolver un problema de horarios con backtracking;
- medir el efecto de MRV, heurística de grado y forward checking;
- diferenciar satisfacción de restricciones y optimización.

![Descripción](https://drive.google.com/uc?export=view&id=1opAXyVoZi8Orca1oSZ3zRk-vG_d868c1)



## 1. Programación de exámenes

Siete exámenes deben asignarse a tres días. Algunos pares comparten estudiantes y no pueden programarse el mismo día.

Un CSP se define por:

- variables $X_1,\dots,X_n$;
- dominios $D_1,\dots,D_n$;
- restricciones que determinan qué asignaciones son válidas.

In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt

VARIABLES = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
DAYS = ['Lunes', 'Martes', 'Miércoles']
CONSTRAINTS = [
    ('A', 'B'), ('A', 'C'), ('B', 'C'), ('B', 'D'), ('B', 'E'),
    ('C', 'E'), ('C', 'F'), ('D', 'E'), ('E', 'F'), ('E', 'G'),
    ('F', 'G')
]

neighbors = defaultdict(set)
for x, y in CONSTRAINTS:
    neighbors[x].add(y)
    neighbors[y].add(x)

print('Vecinos por variable:')
for var in VARIABLES:
    print(var, sorted(neighbors[var]))

## 2. Consistencia

Una asignación parcial es consistente cuando ningún par ya asignado viola una restricción.

In [ ]:
def consistent(assignment):
    for x, y in CONSTRAINTS:
        if x in assignment and y in assignment and assignment[x] == assignment[y]:
            return False
    return True

print(consistent({'A': 'Lunes', 'B': 'Martes'}))
print(consistent({'A': 'Lunes', 'B': 'Lunes'}))

## 3. Backtracking básico

Construimos la asignación una variable a la vez. Cuando una elección conduce a una contradicción, retrocedemos.

In [ ]:
def backtracking_basic(variables, domains):
    stats = {'calls': 0, 'failures': 0}

    def backtrack(assignment):
        stats['calls'] += 1
        if len(assignment) == len(variables):
            return assignment.copy()

        var = next(v for v in variables if v not in assignment)
        for value in domains[var]:
            assignment[var] = value
            if consistent(assignment):
                result = backtrack(assignment)
                if result is not None:
                    return result
            assignment.pop(var)

        stats['failures'] += 1
        return None

    return backtrack({}), stats

domains = {var: DAYS.copy() for var in VARIABLES}
solution_basic, stats_basic = backtracking_basic(VARIABLES, domains)
print('Solución:', solution_basic)
print('Estadísticas:', stats_basic)

## 4. MRV y heurística de grado

**MRV** selecciona la variable no asignada con menos valores disponibles. En caso de empate, la heurística de grado prioriza la variable que restringe a más variables no asignadas.

In [ ]:
def legal_values(var, assignment, domains):
    return [value for value in domains[var]
            if all(assignment.get(n) != value for n in neighbors[var])]


def select_unassigned_variable(assignment, domains):
    unassigned = [v for v in VARIABLES if v not in assignment]
    return min(
        unassigned,
        key=lambda v: (
            len(legal_values(v, assignment, domains)),
            -sum(n not in assignment for n in neighbors[v])
        )
    )

assignment = {'A': 'Lunes'}
print('Siguiente variable:', select_unassigned_variable(assignment, domains))

## 5. Forward Checking

Después de asignar una variable, eliminamos ese valor de los dominios de sus vecinos no asignados. Si algún dominio queda vacío, detectamos el fracaso antes de profundizar.

In [ ]:
def backtracking_with_inference(initial_domains):
    stats = {'calls': 0, 'failures': 0, 'pruned_values': 0}

    def backtrack(assignment, domains):
        stats['calls'] += 1
        if len(assignment) == len(VARIABLES):
            return assignment.copy()

        var = select_unassigned_variable(assignment, domains)
        for value in legal_values(var, assignment, domains):
            new_assignment = assignment.copy()
            new_assignment[var] = value
            new_domains = {v: vals.copy() for v, vals in domains.items()}
            new_domains[var] = [value]

            valid = True
            for neighbor in neighbors[var]:
                if neighbor in new_assignment:
                    continue
                if value in new_domains[neighbor]:
                    new_domains[neighbor].remove(value)
                    stats['pruned_values'] += 1
                if not new_domains[neighbor]:
                    valid = False
                    break

            if valid:
                result = backtrack(new_assignment, new_domains)
                if result is not None:
                    return result

        stats['failures'] += 1
        return None

    return backtrack({}, {v: vals.copy() for v, vals in initial_domains.items()}), stats

solution_fc, stats_fc = backtracking_with_inference(domains)
print('Solución:', solution_fc)
print('Backtracking básico:', stats_basic)
print('MRV + grado + forward checking:', stats_fc)

## 6. Verificación automática

In [ ]:
def validate_solution(solution):
    if solution is None or set(solution) != set(VARIABLES):
        return False
    if any(solution[var] not in DAYS for var in VARIABLES):
        return False
    return all(solution[x] != solution[y] for x, y in CONSTRAINTS)

print('Solución válida:', validate_solution(solution_fc))

## 7. CSP versus optimización

En este problema, cualquier horario consistente es una solución. Si además quisiéramos minimizar el número de días utilizados o equilibrar la cantidad de exámenes por día, agregaríamos una **función objetivo** y el problema se convertiría en una variante de optimización con restricciones.

In [ ]:
counts = {day: list(solution_fc.values()).count(day) for day in DAYS}
plt.figure(figsize=(7, 4))
plt.bar(counts.keys(), counts.values())
plt.ylabel('Número de exámenes')
plt.title('Distribución de la solución encontrada')
plt.grid(True, axis='y')
plt.show()

## Actividades

### Actividad 1 — Dos días

Cambia el dominio para usar solo lunes y martes. ¿Existe solución? Relaciona el resultado con el coloreo de grafos.

In [ ]:
# TODO: prueba el problema con dos días.
two_day_domains = {var: ['Lunes', 'Martes'] for var in VARIABLES}
# solution_two_days, stats_two_days = ...

### Actividad 2 — Restricción unaria

El examen `A` solo puede realizarse el lunes. Modifica su dominio antes de iniciar el backtracking.

In [ ]:
# TODO: crea los dominios con la restricción A = Lunes.
restricted_domains = {var: DAYS.copy() for var in VARIABLES}
# restricted_domains['A'] = ...

### Actividad 3 — Orden de valores

Implementa **Least Constraining Value (LCV)**: prueba primero el valor que elimina menos opciones de los vecinos.

In [ ]:
def order_domain_values(var, assignment, domains):
    # TODO: retorna los valores legales ordenados de menor a mayor impacto.
    pass

### Actividad 4 — Optimización sobre el CSP

Entre todas las soluciones válidas, busca una que minimice el desbalance:

$$
\max_d n_d-\min_d n_d
$$

where $n_d$ is the number of exams assigned to day $d$. Puedes enumerar todas las soluciones con backtracking y conservar la mejor.

## Preguntas de cierre

1. ¿Qué información utiliza MRV para seleccionar una variable?
2. ¿Forward checking evita completamente el backtracking?
3. ¿Una solución consistente parcial es necesariamente extensible a una solución completa?
4. ¿Cuándo un CSP se convierte en un problema de optimización?